# Post Race Evaluation

Input
- constructor and driver prediction tables
- hist driver and constructor points
- team config table (not yet created)


Processing
- log prediction absolute errors
- log best team (using teamOptimizer notebook/script)

Output
- show graph of trends of best team as well as my error rate

### Libraries

In [1661]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

import sys
sys.path.append("/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject")

import teamOptimizer as to

In [1662]:
import os
os.getcwd()

'/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject/notebooks/evaluation'

In [1663]:
working_directory = Path.cwd().parent.parent
print(working_directory)

/Users/jackguptill/Library/CloudStorage/OneDrive-Personal/Code/F1FantasyProject


### Data Import

In [1664]:
driver_path = working_directory / "data" / "predictions" / "drivers" / "driver_predictions_2026.csv"
con_path = working_directory / "data" / "predictions" / "constructors" / "constructor_preds_2026.csv"

driver_race_results_path = working_directory / "data" / "clean" / "driver_points.csv"
constructor_race_results_path = working_directory / "data" / "clean" / "constructor_points.csv"

team_path = working_directory / "data" / "myTeam" / "teamConfig.csv"

#prediction tables
driver_preds = pd.read_csv(driver_path).drop(columns=['Unnamed: 0'])
constructor_preds = pd.read_csv(con_path)

#results tables
driver_race_results = pd.read_csv(driver_race_results_path).drop(columns=['Unnamed: 0'])
con_race_results = pd.read_csv(constructor_race_results_path).drop(columns=['Unnamed: 0'])

#weekly config table
team_config = pd.read_csv(team_path)

## Data Clean

- joining and cleaning tables to get a single table that has our prediction performance
- will eventually use this to create graphs on how we are doing

### Creating a "Log" of the performance of predictions and results
- will contain features such as absolute error rate

In [1665]:
constructor_preds["asset_name"] = constructor_preds["constructor"]
con_race_results["asset_name"] = con_race_results["constructor"]
con_race_results = con_race_results.drop(columns=["constructor"])

In [1666]:
driver_preds["asset_name"] = driver_preds["driver"]
driver_race_results["asset_name"] = driver_race_results["driver"]
driver_preds = driver_preds.drop(columns=["driver"])

In [1667]:
#name change to be in sync with preds
driver_race_results["race_num"] = driver_race_results["race"]
con_race_results["race_num"] = con_race_results["race"]

driver_race_results = driver_race_results.drop(columns=["race"])
con_race_results = con_race_results.drop(columns=["race"])

In [1668]:
constructor_preds["asset_type"] = "constructor"

In [1669]:
driver_preds["asset_type"] = "driver"

In [1670]:
predictions = pd.concat([constructor_preds, driver_preds], axis=0, ignore_index = True)
predictions["asset_name"] = predictions['asset_name'].str.upper()
predictions = predictions[["year", "race_num", "asset_name", "price", "predicted_points", "asset_type"]]
predictions

,year,race_num,asset_name,price,predicted_points,asset_type
0,2026,1,MCL,28.9,53.242756,constructor
1,2026,1,MER,29.3,51.584534,constructor
2,2026,1,RBR,28.2,48.359364,constructor
3,2026,1,FER,23.3,40.582527,constructor
4,2026,1,RB,6.3,26.878194,constructor
...,...,...,...,...,...,...
61,2026,1,BOR,6.4,5.940938,driver
62,2026,1,COL,6.2,8.773124,driver
63,2026,1,LIN,6.2,2.862940,driver
64,2026,1,PER,6.0,3.668579,driver


## Left Join Results to the predictions

In [1671]:
driver_race_results = driver_race_results[['year', 'race_num', 'asset_name', 'points']]
con_race_results = con_race_results[["year", "race_num", "asset_name", "points"]]
predictions = predictions.merge(driver_race_results, how = "left", on=["year", "race_num", "asset_name"])
predictions = predictions.merge(con_race_results, how = "left", on=["year", "race_num", "asset_name"])

predictions["points"] = predictions["points_x"].combine_first(predictions["points_y"])
predictions = predictions.drop(columns=["points_x", "points_y"])
predictions = predictions.drop_duplicates(subset=["year", "race_num", "asset_name"])
predictions

,year,race_num,asset_name,price,predicted_points,asset_type,points
0,2026,1,MCL,28.9,53.242756,constructor,19.0
1,2026,1,MER,29.3,51.584534,constructor,96.0
2,2026,1,RBR,28.2,48.359364,constructor,42.0
3,2026,1,FER,23.3,40.582527,constructor,69.0
4,2026,1,RB,6.3,26.878194,constructor,35.0
5,2026,1,AMR,10.3,22.922543,constructor,-38.0
6,2026,1,WIL,12.0,22.280189,constructor,20.0
7,2026,1,HAS,7.4,19.857089,constructor,34.0
8,2026,1,ALP,12.5,16.529915,constructor,22.0
9,2026,1,AUD,6.6,10.462140,constructor,0.0


#### creating a clean copy for visualizations and metrics

In [1672]:
performance = predictions.copy()

## Getting Our Overall Mean Absolute Error of Our Predictions

In [1673]:
performance["absolute_error"] = abs(performance["points"] - performance['predicted_points'])


In [1674]:
performance = performance.sort_values(["absolute_error"], ascending=False)
performance

,year,race_num,asset_name,price,predicted_points,asset_type,points,absolute_error
5,2026,1,AMR,10.3,22.922543,constructor,-38.0,60.922543
1,2026,1,MER,29.3,51.584534,constructor,96.0,44.415466
0,2026,1,MCL,28.9,53.242756,constructor,19.0,34.242756
25,2026,1,PIA,25.5,16.833676,driver,-14.0,30.833676
37,2026,1,HUL,6.8,9.227555,driver,-20.0,29.227555
3,2026,1,FER,23.3,40.582527,constructor,69.0,28.417473
34,2026,1,STR,8.0,3.485443,driver,-23.0,26.485443
10,2026,1,CAD,6.0,9.292076,constructor,-13.0,22.292076
29,2026,1,HAD,15.1,13.011118,driver,-8.0,21.011118
43,2026,1,BOT,5.9,4.211683,driver,-16.0,20.211683


In [1675]:
np.mean(performance["absolute_error"])

np.float64(15.238592298534043)

In [1676]:
#FIXME: make a current week or past week MAE

## What was the Optimal Team Last Week?

#### re-reading in a clean version of the hist tables for this

In [1677]:
driver_roster_path = working_directory / "data" / "clean" / "driver_roster.csv"
last_week_construcor_pricing_path = working_directory / "data" / "clean" / "constructor_price.csv"
last_week_driver_pricing_path = working_directory / "data" / "clean" / "driver_price.csv"


last_week_drivers = pd.read_csv(driver_race_results_path).drop(columns=['Unnamed: 0'])
last_week_constructors = pd.read_csv(constructor_race_results_path).drop(columns=['Unnamed: 0'])
last_week_driver_pricing = pd.read_csv(last_week_driver_pricing_path).drop(columns=['Unnamed: 0'])
last_week_construcor_pricing = pd.read_csv(last_week_construcor_pricing_path).drop(columns=['Unnamed: 0'])
last_week_driver_roster = pd.read_csv(driver_roster_path).drop(columns=['Unnamed: 0'])

In [1678]:
latest_row = last_week_drivers.sort_values(["year", "race"]).iloc[-1]
year = latest_row["year"]
race = latest_row["race"]
print(year, race)

2026 1


#### Filtering to latest week

In [1679]:
last_week_drivers = last_week_drivers[(last_week_drivers["year"] == year) & (last_week_drivers["race"] == race)].reset_index(drop=True)
last_week_constructors = last_week_constructors[(last_week_constructors["year"] == year) & (last_week_constructors["race"] == race)].reset_index(drop=True)
last_week_driver_roster = last_week_driver_roster[(last_week_driver_roster["year"] == year) & (last_week_driver_roster["race"] == race)].reset_index(drop=True)
last_week_driver_pricing = last_week_driver_pricing[(last_week_driver_pricing["year"] == year) & (last_week_driver_pricing["race"] == race)].reset_index(drop=True)
last_week_construcor_pricing = last_week_construcor_pricing[(last_week_construcor_pricing["year"] == year) & (last_week_construcor_pricing["race"] == race)].reset_index(drop=True)



In [1680]:
last_week_construcor_pricing.head()

,constructor,race,price,year,race_name,start_date,location,country_code,month,start_epoch
0,MER,1,29.3,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400
1,MCL,1,28.9,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400
2,RBR,1,28.2,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400
3,FER,1,23.3,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400
4,ALP,1,12.5,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400


In [1681]:
last_week_driver_pricing = last_week_driver_pricing[["year", "race", "driver", "price"]]
last_week_constructor_pricing = last_week_construcor_pricing[["year", "race", "constructor", "price"]]

In [1682]:
#driver merges
last_week_drivers = last_week_drivers.merge(last_week_driver_roster, on=["year", "race", "driver"], how="left")
last_week_drivers = last_week_drivers.merge(last_week_driver_pricing, on=["year", "race", "driver"], how="left")


last_week_drivers

,driver,race,points,year,race_name,start_date,location,country_code,month,start_epoch,constructor,price
0,VER,1,50.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,RBR,27.7
1,RUS,1,39.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,MER,27.4
2,NOR,1,21.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,MCL,27.2
3,PIA,1,-14.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,MCL,25.5
4,ANT,1,32.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,MER,23.2
5,LEC,1,29.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,FER,22.8
6,HAM,1,25.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,FER,22.5
7,HAD,1,-8.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,RBR,15.1
8,GAS,1,11.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,ALP,12.0
9,SAI,1,9.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,WIL,11.8


In [1683]:
#constructor merges
last_week_constructors = last_week_constructors.merge(last_week_constructor_pricing, on=["year", "race", "constructor"], how="left")

In [1684]:
#last week driver table build
#cols needed: year, race_num, driver, price, constructor, points
#renaming as needed
last_week_drivers = last_week_drivers.rename(columns={'race':'race_num'})
last_week_drivers = last_week_drivers[["year", "race_num", "driver", "price", "points", "constructor"]]


In [1685]:
last_week_constructors

,constructor,race,points,year,race_name,start_date,location,country_code,month,start_epoch,price
0,MER,1,96.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,29.3
1,MCL,1,19.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,28.9
2,RBR,1,42.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,28.2
3,FER,1,69.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,23.3
4,ALP,1,22.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,12.5
5,WIL,1,20.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,12.0
6,AMR,1,-38.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,10.3
7,HAS,1,34.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,7.4
8,AUD,1,0.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,6.6
9,RB,1,35.0,2026,Australian Grand Prix,2026-03-08 04:00:00+00:00,Melbourne,AUS,3,1772942400,6.3


In [1686]:
#last week constructor table build
#cols needed: year, race_num, constructor, price, points
last_week_constructors = last_week_constructors.rename(columns={'race':'race_num'})
last_week_constructors = last_week_constructors[["year", "race_num", "constructor", "price", "points"]]

In [1688]:
drivers_sel, cons_sel, summary = to.optimize_team(
    budget=100,
    drivers=last_week_drivers,
    cons=last_week_constructors,
    points_col="points",
    n_drivers=5,
    n_constructors=2,
    max_drivers_per_team=2,
    use_drs=True,
    drs_multiplier=2.0,
    solver_msg=False,
    require_driver_from_each_constructor=False,
    min_drivers_per_selected_constructor=1
)

print(summary)
display(drivers_sel)
display(cons_sel)

{'status': 1, 'budget': 100, 'points_column_used': 'points', 'total_price': 95.3, 'total_points': 290.0, 'drs_driver': 'VER'}


,year,race_num,driver,price,points,constructor,team_key
0,2026,1,VER,27.7,50.0,RBR,RBR
1,2026,1,BEA,7.4,20.0,HAS,HAS
2,2026,1,LIN,6.2,15.0,RB,RB
3,2026,1,BOR,6.4,13.0,AUD,AUD
4,2026,1,GAS,12.0,11.0,ALP,ALP


,year,race_num,constructor,price,points,team_key
0,2026,1,MER,29.3,96.0,MER
1,2026,1,RB,6.3,35.0,RB


### Log the Best Driver Config

In [1691]:
drivers_sel = drivers_sel.rename(columns={"driver": "asset_name"})
cons_sel = cons_sel.rename(columns={"constructor": "asset_name"})

drivers_sel["asset_type"] = "driver"
cons_sel["asset_type"] = "constructor"

In [1692]:
cols = [
    "year",
    "race_num",
    "asset_name",
    "asset_type",
    "price",
    "points",
    "team_key"
]

drivers_sel = drivers_sel[cols]
cons_sel = cons_sel[cols]

In [1693]:
best_team = pd.concat([drivers_sel, cons_sel], ignore_index=True)

In [1695]:
best_team["is_drs"] = np.where(
    best_team["asset_name"] == summary["drs_driver"],
    "yes",
    "no"
)

In [1697]:
best_team.head()

,year,race_num,asset_name,asset_type,price,points,team_key,is_drs
0,2026,1,VER,driver,27.7,50.0,RBR,yes
1,2026,1,BEA,driver,7.4,20.0,HAS,no
2,2026,1,LIN,driver,6.2,15.0,RB,no
3,2026,1,BOR,driver,6.4,13.0,AUD,no
4,2026,1,GAS,driver,12.0,11.0,ALP,no


### append data to best_team log

In [1698]:
best_team_file_path = working_directory /'data' / 'myTeam' / 'bestTeam.csv'
best_team.to_csv(best_team_file_path, mode="a", header=False, index=False)